## Peer-to-Peer Communication: Switching the Active Policy

Two local policies peer over localhost WebSockets.  After peering,
the active policy is switched to the remote proxy so that subsequent
calls transparently operate on the remote policy.

### What this notebook does
1. Create two independent policies (**Alice** and **Bob**).
2. Register an extra pool on Bob and memorize an entry locally on Bob.
3. Start Bob's communication server and peer Alice to Bob.
4. From Alice's console, switch the active policy to the **remote Bob proxy**.
5. Query Bob's memory remotely — verify the entry exists through the proxy.
6. Demonstrate bidirectional RPC: Bob calls a method on Alice too.
7. Switch back to Alice and clean up.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time
import laila
from laila.policy.schema.base import _LAILA_IDENTIFIABLE_POLICY
from laila.pool.schema.base import _LAILA_IDENTIFIABLE_POOL

### 1 — Create two policies

In [ ]:
alice = _LAILA_IDENTIFIABLE_POLICY()
bob   = _LAILA_IDENTIFIABLE_POLICY()

# Wire local_policy so inbound RPCs can traverse the policy tree
alice.central.communication._local_policy = alice
bob.central.communication._local_policy   = bob

print("Alice:", alice.global_id)
print("Bob  :", bob.global_id)

### 2 — Add a pool on Bob and memorize an entry locally

In [ ]:
bob_pool = _LAILA_IDENTIFIABLE_POOL()
bob.central.memory.add_pool(pool=bob_pool, affinity=1.0, pool_nickname="bob-store")

# Work as Bob locally to memorize an entry
laila.activate_policy(bob)
entry = laila.constant(data={"message": "stored by Bob locally"}, nickname="bob-local-entry")
future = laila.memorize(entries=entry, pool_nickname="bob-store")
laila.wait(future)

print("Entry:", entry.global_id)
print("Bob pool has entry:", entry.global_id in bob_pool.resource)

### 3 — Peer Alice → Bob over localhost

In [ ]:
bob.central.communication.start()
bob_port   = bob.central.communication.port
bob_secret = bob.central.communication.peer_secret_key

remote_bob_id = alice.central.communication.add_peer(
    f"ws://127.0.0.1:{bob_port}", bob_secret
)
time.sleep(0.3)

print(f"Peered successfully.")
print(f"Alice peers: {list(alice.central.communication.peers)}")
print(f"Bob   peers: {list(bob.central.communication.peers)}")

### 4 — Switch active policy to remote Bob proxy

From Alice's perspective, activating the remote proxy means all
subsequent `laila.*` calls route to Bob over the WebSocket.

In [ ]:
laila.activate_policy(alice)
print("Active policy:", laila.get_active_policy().global_id, "(Alice)")

# Grab the remote proxy and switch
remote_bob = laila.peers[bob.global_id]
laila.activate_policy(remote_bob)
print("Active policy:", laila.get_active_policy().global_id, "(Bob via proxy)")

### 5 — Query Bob's memory remotely from Alice's console

Even though we are on Alice's machine, the proxy transparently
traverses Bob's policy tree over the wire.

In [ ]:
# Ask Bob's pool router for the pool registered under "bob-store"
routed_pool = remote_bob.central.memory.pool_router.route(
    entries=[entry.global_id], pool_nickname="bob-store"
)
print("Remote route returned:", type(routed_pool).__name__)
print("Pool data (serialized via RPC):")
for k, v in routed_pool.items():
    if k == "resource":
        print(f"  resource: {len(v)} entries stored")
    else:
        print(f"  {k}: {v}")

### 6 — Bidirectional: Bob calls a method on Alice

Peering is fully symmetric — Bob can also call into Alice.

In [ ]:
# Add a pool to Alice
alice_pool = _LAILA_IDENTIFIABLE_POOL()
alice.central.memory.add_pool(pool=alice_pool, affinity=1.0, pool_nickname="alice-store")

# From Bob's side, call into Alice via proxy
remote_alice = bob.central.communication.peers[alice.global_id]
alice_routed = remote_alice.central.memory.pool_router.route(
    entries=["any-id"], pool_nickname="alice-store"
)
print("Bob queried Alice's pool router via RPC.")
print("Alice pool batch_accelerated:", alice_routed.get("batch_accelerated"))

### 7 — Switch back to Alice and clean up

In [ ]:
laila.activate_policy(alice)
print("Active policy:", laila.get_active_policy().global_id, "(Alice)")

alice.central.communication.stop()
bob.central.communication.stop()
print("Connections closed.")